# Inspeção Inicial dos Dados

> **Objetivo:** Validar premissas sobre os dados antes de iniciar a análise exploratória.
> Este notebook verifica schemas, chaves de join, missing values e distribuições básicas.

**Dados inspecionados:**
- `whatsapp_base_disparo_mascarado.parquet` — log de disparos
- `whatsapp_dim_telefone_mascarado.parquet` — catálogo de telefones com metadados

**⚠️ ATENÇÃO:** O schema documentado (`whatsapp_schema.yml`) possui divergências em relação aos dados reais. Os achados estão documentados na Seção 4.

---
## 01. Import das Bilbiotecas e Configurações iniciais


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

import requests
from pathlib import Path

# Configurações visuais
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Bibliotecas carregadas com sucesso.')


Bibliotecas carregadas com sucesso.


---
## 02. Verificação dos Dados e Download se necessário

> **Nota:** Esta seção verifica se os arquivos Parquet existem localmente. Caso não existam, realiza o download automático do bucket GCS público.

**Bucket GCS:** `gs://case_vagas/whatsapp/`

**Arquivos no bucket:**
- `base_disparo_mascarado` (sem extensão `.parquet` no bucket)
- `dim_telefone_mascarado` (sem extensão `.parquet` no bucket)
- `schema.yml`

**Arquivos locais esperados:**
- `whatsapp_base_disparo_mascarado.parquet`
- `whatsapp_dim_telefone_mascarado.parquet`
- `whatsapp_schema.yml`

---

In [ ]:
BASE_DIR = Path('.').resolve().parent
DATA_DIR = BASE_DIR / 'data' / 'raw'

PATH_DISPARO = DATA_DIR / 'whatsapp_base_disparo_mascarado.parquet'
PATH_TELEFONE = DATA_DIR / 'whatsapp_dim_telefone_mascarado.parquet'

print(f'Diretório base: {BASE_DIR}')
print(f'Diretório de dados: {DATA_DIR}')
print(f'Arquivo de disparos existe: {PATH_DISPARO.exists()}')
print(f'Arquivo de telefones existe: {PATH_TELEFONE.exists()}')


Diretório base: C:\Users\deand\Desktop\CODE\CTP_desafio\desafio-cientista-dados-pleno-campanhas
Diretório de dados: C:\Users\deand\Desktop\CODE\CTP_desafio\desafio-cientista-dados-pleno-campanhas\data\raw
Arquivo de disparos existe: True
Arquivo de telefones existe: True


In [3]:
# DOWNLOAD DO GCS

GCS_BASE_URL = 'https://storage.googleapis.com/case_vagas/whatsapp'

# Usar o mesmo BASE_DIR definido na célula anterior
# (garante consistência independente de onde o notebook é executado)
DATA_DIR = BASE_DIR / 'data' / 'raw'

# Garantir que o diretório existe
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento: nome_local -> nome_no_bucket
ARQUIVOS = {
    'whatsapp_base_disparo_mascarado.parquet': 'base_disparo_mascarado',
    'whatsapp_dim_telefone_mascarado.parquet': 'dim_telefone_mascarado',
    'whatsapp_schema.yml': 'schema.yml',
}

def download_do_gcs(nome_local, nome_bucket):
    """
    Baixa um arquivo do bucket GCS público caso não exista localmente.
    Retorna True se o download foi necessário, False se já existia.
    """
    caminho_local = DATA_DIR / nome_local
    
    if caminho_local.exists():
        print(f'  ✅ {nome_local} já existe ({caminho_local.stat().st_size:,} bytes)')
        return False
    
    url = f'{GCS_BASE_URL}/{nome_bucket}'
    print(f'  ⬇️  Baixando {nome_local} de {url} ...')
    
    try:
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()
        
        with open(caminho_local, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f'  ✅ Download concluído ({caminho_local.stat().st_size:,} bytes)')
        return True
    except requests.exceptions.RequestException as e:
        print(f'  ❌ Erro ao baixar {nome_local}: {e}')
        # Remover arquivo parcial se existir
        if caminho_local.exists():
            caminho_local.unlink()
        raise

# Executar downloads
print('=== VERIFICANDO/BAXANDO DADOS DO GCS ===')
downloads_realizados = 0
for nome_local, nome_bucket in ARQUIVOS.items():
    if download_do_gcs(nome_local, nome_bucket):
        downloads_realizados += 1

print(f'\nResumo: {downloads_realizados} download(s) realizado(s), {len(ARQUIVOS) - downloads_realizados} já existia(m).')


=== VERIFICANDO/BAXANDO DADOS DO GCS ===
  ✅ whatsapp_base_disparo_mascarado.parquet já existe (19,965,833 bytes)
  ✅ whatsapp_dim_telefone_mascarado.parquet já existe (18,420,043 bytes)
  ✅ whatsapp_schema.yml já existe (3,572 bytes)

Resumo: 0 download(s) realizado(s), 3 já existia(m).


---
## 03. Carregamento dos Dados e Divergências do Schema

Carregamos os dados e já documentamos as divergências encontradas em relação ao `whatsapp_schema.yml`:

| Schema (YAML) | Realidade (Parquet) | Ação |
|---|---|---|
| Coluna `telefone_mascarado` na dim | **Não existe**. A chave é `telefone_numero` | Usar `telefone_numero` como join |
| `status_disparo`: DELIVERED, READ, FAILED, SENT | Existe também **`processing`** | Documentar 5 status |
| Colunas `resposta_datahora`, `fim_datahora` | **Não existem** no parquet | Ignorar |
| — | Existem colunas extras: `indicador_falha`, `id_status_disparo` | Considerar se úteis |

> **Nota:** A chave de join correta é: `contato_telefone` (base) == `telefone_numero` (dim). Isso foi validado pela magnitude idêntica dos valores (min/max iguais) e match parcial (~252k telefones).

In [4]:
# Carregar dados completos (parquet é eficiente em memória)
df_disparo = pd.read_parquet(PATH_DISPARO)
df_telefone = pd.read_parquet(PATH_TELEFONE)

print('=== BASE DE DISPAROS ===')
print(f'Shape: {df_disparo.shape}')
print(f'Colunas: {list(df_disparo.columns)}')
print()
print('=== DIMENSÃO DE TELEFONES ===')
print(f'Shape: {df_telefone.shape}')
print(f'Colunas: {list(df_telefone.columns)}')
print()
print('Divergências do schema confirmadas.')


=== BASE DE DISPAROS ===
Shape: (392921, 16)
Colunas: ['id_conta', 'id_hsm', 'id_disparo', 'id_sessao', 'cpf', 'id_target', 'contato_telefone', 'categoria_hsm', 'ambiente', 'criacao_envio_datahora', 'envio_datahora', 'falha_datahora', 'descricao_falha', 'indicador_falha', 'id_status_disparo', 'status_disparo']

=== DIMENSÃO DE TELEFONES ===
Shape: (283289, 11)
Colunas: ['telefone_ddi', 'telefone_ddd', 'telefone_numero', 'telefone_tipo', 'telefone_nacionalidade', 'telefone_qualidade', 'telefone_aparicoes', 'telefone_aparicoes_quantidade', 'telefone_proprietarios_quantidade', 'telefone_sistemas_quantidade', 'validacao_telefone']

Divergências do schema confirmadas.


---
## 04. Verificação 1 — Shapes, DTypes e Memory Usage

Validamos se os tipos de dados estão consistentes (especialmente datas e arrays).

In [5]:
print('=== INFO: BASE DE DISPAROS ===')
df_disparo.info()
print('\n=== INFO: DIMENSÃO DE TELEFONES ===')
df_telefone.info()

=== INFO: BASE DE DISPAROS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392921 entries, 0 to 392920
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   id_conta                392921 non-null  int64         
 1   id_hsm                  392921 non-null  int64         
 2   id_disparo              392921 non-null  int64         
 3   id_sessao               236904 non-null  float64       
 4   cpf                     361745 non-null  float64       
 5   id_target               392921 non-null  int64         
 6   contato_telefone        392921 non-null  int64         
 7   categoria_hsm           392921 non-null  object        
 8   ambiente                392921 non-null  object        
 9   criacao_envio_datahora  392921 non-null  datetime64[us]
 10  envio_datahora          392921 non-null  datetime64[us]
 11  falha_datahora          27261 non-null   datetime64[us]
 12 

---
## 05. Verificação 2 — Missing Values

Focamos especialmente nas colunas de join (`contato_telefone`, `telefone_numero`) e no array `telefone_aparicoes`.

In [6]:
def missing_report(df, nome):
    """Exibe relatório de missing values ordenado por % de nulos."""
    missing = df.isnull().sum()
    pct = 100 * missing / len(df)
    report = pd.DataFrame({
        'coluna': df.columns,
        'nulos': missing.values,
        'pct_nulos': pct.values
    }).sort_values('pct_nulos', ascending=False)
    print(f'=== MISSING VALUES: {nome} ===')
    display(report[report['nulos'] > 0])
    print()

missing_report(df_disparo, 'BASE DISPARO')
missing_report(df_telefone, 'DIM TELEFONE')


=== MISSING VALUES: BASE DISPARO ===


,coluna,nulos,pct_nulos
12,descricao_falha,365660,93.061964
11,falha_datahora,365660,93.061964
3,id_sessao,156017,39.706964
4,cpf,31176,7.934419



=== MISSING VALUES: DIM TELEFONE ===


,coluna,nulos,pct_nulos


---
## 06. Verificação 3 — Valores Únicos de Status de Disparo

O schema mencionava: `DELIVERED`, `READ`, `FAILED`, `SENT`. Confirmamos que existe também `processing`.

In [7]:
print('Status únicos na base de disparos:')
print(df_disparo['status_disparo'].value_counts(dropna=False))

# Verificar se há variação de caixa (upper/lower/title case)
statuses = df_disparo['status_disparo'].dropna().unique()
print(f'\nValores brutos: {statuses}')


Status únicos na base de disparos:
status_disparo
read          271566
delivered      85405
failed         27257
sent            5533
processing      3160
Name: count, dtype: int64

Valores brutos: ['processing' 'sent' 'delivered' 'read' 'failed']


---
## 07. Verificação 4 — Compatibilidade da Chave de Join

**Chave validada:** `contato_telefone` (base) == `telefone_numero` (dim).

Verificamos:
1. Se os valores são do mesmo tipo
2. Quantos telefones da base NÃO têm correspondência na dimensão (unmatched)
3. Quantos telefones da dimensão NUNCA foram usados em disparos

In [8]:
# Tipos das colunas de join
print(f'Tipo de contato_telefone: {df_disparo["contato_telefone"].dtype}')
print(f'Tipo de telefone_numero: {df_telefone["telefone_numero"].dtype}')

# Exemplos de valores
print('\nExemplo contato_telefone:', df_disparo['contato_telefone'].iloc[0])
print('Exemplo telefone_numero:', df_telefone['telefone_numero'].iloc[0])

# Magnitude (min/max) para confirmar se são a mesma escala
print('\nMagnitude contato_telefone:', df_disparo['contato_telefone'].abs().min(), 'a', df_disparo['contato_telefone'].abs().max())
print('Magnitude telefone_numero:', df_telefone['telefone_numero'].abs().min(), 'a', df_telefone['telefone_numero'].abs().max())

# Set intersection para verificar match
telefones_disparo = set(df_disparo['contato_telefone'].dropna().unique())
telefones_dim = set(df_telefone['telefone_numero'].dropna().unique())

intersecao = telefones_disparo & telefones_dim
apenas_disparo = telefones_disparo - telefones_dim
apenas_dim = telefones_dim - telefones_disparo

print(f'\nTelefones únicos na base de disparos: {len(telefones_disparo):,}')
print(f'Telefones únicos na dimensão: {len(telefones_dim):,}')
print(f'Match (interseção): {len(intersecao):,}')
print(f'Apenas em disparos (sem match na dim): {len(apenas_disparo):,} ({100*len(apenas_disparo)/len(telefones_disparo):.2f}%)')
print(f'Apenas na dim (nunca disparados): {len(apenas_dim):,} ({100*len(apenas_dim)/len(telefones_dim):.2f}%)')


Tipo de contato_telefone: int64
Tipo de telefone_numero: int64

Exemplo contato_telefone: 2824089259510570290
Exemplo telefone_numero: -6862804366069381626

Magnitude contato_telefone: 21129070184792 a 9223341837253659603
Magnitude telefone_numero: 21129070184792 a 9223341837253659603

Telefones únicos na base de disparos: 294,903
Telefones únicos na dimensão: 283,289
Match (interseção): 252,497
Apenas em disparos (sem match na dim): 42,406 (14.38%)
Apenas na dim (nunca disparados): 30,792 (10.87%)


---
## 08. Verificação 5 — Estrutura de `telefone_aparicoes`

Este é o campo mais crítico. Precisamos confirmar:
- É de fato um array/lista de dicionários?
- Quais chaves existem em cada dicionário?
- Há valores nulos ou arrays vazios?
- O nome do campo de sistema é `id_sistema` ou `id_sistema_mask`?

In [9]:
# Inspecionar estrutura do campo telefone_aparicoes
exemplo = df_telefone['telefone_aparicoes'].iloc[0]
print(f'Tipo do campo: {type(exemplo)}')
print(f'Exemplo bruto: {exemplo}')

# Verificar arrays vazios ou nulos
nulos = df_telefone['telefone_aparicoes'].isnull().sum()
vazios = df_telefone['telefone_aparicoes'].apply(lambda x: len(x) == 0 if isinstance(x, (list, np.ndarray)) else False).sum()
print(f'\nArrays nulos: {nulos}')
print(f'Arrays vazios: {vazios}')

# Explorar chaves dos dicionários internos
primeiro_nao_vazio = None
for val in df_telefone['telefone_aparicoes']:
    if isinstance(val, (list, np.ndarray)) and len(val) > 0:
        primeiro_nao_vazio = val
        break

if primeiro_nao_vazio:
    print(f'\nPrimeiro item do array: {primeiro_nao_vazio[0]}')
    print(f'Tipo do primeiro item: {type(primeiro_nao_vazio[0])}')
    if isinstance(primeiro_nao_vazio[0], dict):
        print(f'Chaves do dict: {list(primeiro_nao_vazio[0].keys())}')

# Estatísticas de quantidade de aparições por telefone
aparicoes_len = df_telefone['telefone_aparicoes'].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
print(f'\nEstatísticas de quantidade de sistemas por telefone:')
print(aparicoes_len.describe())


Tipo do campo: <class 'numpy.ndarray'>
Exemplo bruto: [{'id_sistema': '1257277410380486863', 'cpf': 5073517428359850284, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': datetime.date(2024, 10, 30)}]

Arrays nulos: 0
Arrays vazios: 0

Primeiro item do array: {'id_sistema': '1257277410380486863', 'cpf': 5073517428359850284, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': datetime.date(2024, 10, 30)}
Tipo do primeiro item: <class 'dict'>
Chaves do dict: ['id_sistema', 'cpf', 'proprietario_tipo', 'registro_data_atualizacao']

Estatísticas de quantidade de sistemas por telefone:
count    283289.000000
mean          5.400040
std         210.982556
min           1.000000
25%           2.000000
50%           3.000000
75%           6.000000
max       76480.000000
Name: telefone_aparicoes, dtype: float64


---
## 09. Verificação 6 — Distribuições Rápidas

Análise superficial de variáveis categóricas e numéricas importantes.

### 9.1 Base de Disparos

In [10]:
print('=== CATEGORIA HSM ===')
print(df_disparo['categoria_hsm'].value_counts().head(10))

print('\n=== AMBIENTE ===')
print(df_disparo['ambiente'].value_counts())

print('\n=== DESCRIÇÃO DE FALHA (top 10) ===')
print(df_disparo['descricao_falha'].value_counts().head(10))

# Range de datas
print('\n=== RANGE DE DATAS ===')
for col in ['envio_datahora', 'falha_datahora']:
    if col in df_disparo.columns:
        print(f'{col}: {df_disparo[col].min()} a {df_disparo[col].max()}')


=== CATEGORIA HSM ===
categoria_hsm
Utilidade       362389
Autenticação     30439
Marketing           86
REMOVED              7
Name: count, dtype: int64

=== AMBIENTE ===
ambiente
prod       392876
dev            31
hom             7
REMOVED         7
Name: count, dtype: int64

=== DESCRIÇÃO DE FALHA (top 10) ===
descricao_falha
CÓDIGO: 131026. ERRO: A mensagem não pode ser entregue. O número de telefone do destinatário não está registrado no WhatsApp ou o destinatário não aceitou os novos Termos de Serviço e a nova Política de Privacidade.    26649
Falha ao processar o envio do HSM                                                                                                                                                                                            412
Erro desconhecido.                                                                                                                                                                                                        

### 9.2 Dimensão de Telefones

In [11]:

print('=== TELEFONE TIPO ===')
print(df_telefone['telefone_tipo'].value_counts())

print('\n=== TELEFONE QUALIDADE ===')
print(df_telefone['telefone_qualidade'].value_counts())

print('\n=== TELEFONE NACIONALIDADE ===')
print(df_telefone['telefone_nacionalidade'].value_counts())

print('\n=== PROPRIETÁRIOS QUANTIDADE (top 10) ===')
print(df_telefone['telefone_proprietarios_quantidade'].value_counts().sort_index().head(10))

print('\n=== SISTEMAS QUANTIDADE (top 10) ===')
print(df_telefone['telefone_sistemas_quantidade'].value_counts().sort_index().head(10))

print('\n=== TOP 10 DDDs ===')
print(df_telefone['telefone_ddd'].value_counts().head(10))


=== TELEFONE TIPO ===
telefone_tipo
CELULAR    283281
FIXO            8
Name: count, dtype: int64

=== TELEFONE QUALIDADE ===
telefone_qualidade
VALIDO      241889
SUSPEITO     40673
INVALIDO       727
Name: count, dtype: int64

=== TELEFONE NACIONALIDADE ===
telefone_nacionalidade
Brasil    283289
Name: count, dtype: int64

=== PROPRIETÁRIOS QUANTIDADE (top 10) ===
telefone_proprietarios_quantidade
1     77073
2     65080
3     46855
4     32000
5     20890
6     13408
7      8729
8      6173
9      4051
10     2957
Name: count, dtype: int64

=== SISTEMAS QUANTIDADE (top 10) ===
telefone_sistemas_quantidade
1     88371
2    104276
3     68124
4     21168
5      1330
6        20
Name: count, dtype: int64

=== TOP 10 DDDs ===
telefone_ddd
-1181433720517268842    278066
 6906343888316397859      1112
 1110362451252208393       969
-2600719053607284362       412
-4523905025782287081       260
 2685248380958544781       198
-8921310246710213502       187
-2978013433900144088       129
-573

### 9.3 Validação do Telefone (dict interno)

In [12]:

exemplo_validacao = df_telefone['validacao_telefone'].iloc[0]
print(f'Tipo do campo validacao_telefone: {type(exemplo_validacao)}')
print(f'Exemplo: {exemplo_validacao}')

if isinstance(exemplo_validacao, dict):
    print(f'Chaves disponíveis: {list(exemplo_validacao.keys())}')


Tipo do campo validacao_telefone: <class 'dict'>
Exemplo: {'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}
Chaves disponíveis: ['ddd_valido_br', 'formato_valido', 'padrao_suspeito', 'padrao_invalido']


---
## 10. Verificação Extra — CPFs na dimensão vs. CPFs na base de disparos

Um CPF pode ter vários telefones. Verificamos se há CPFs na dimensão que nunca foram disparados.

In [13]:
# Extrair CPFs da dimensão (precisamos explodir telefone_aparicoes para acessar os CPFs)
# Primeiro, verificar se há coluna cpf direta na dimensão
print('Colunas da dimensão:', df_telefone.columns.tolist())

# A dimensão não tem coluna cpf direta; os CPFs estão dentro de telefone_aparicoes
# Fazemos uma extração rápida para estimar a cobertura
cpfs_dim = set()
for aparicoes in df_telefone['telefone_aparicoes']:
    if isinstance(aparicoes, (list, np.ndarray)):
        for item in aparicoes:
            if isinstance(item, dict) and 'cpf' in item:
                cpfs_dim.add(item['cpf'])

cpfs_disparo = set(df_disparo['cpf'].dropna().unique())

print(f'\nCPFs únicos na base de disparos: {len(cpfs_disparo):,}')
print(f'CPFs únicos na dimensão (via aparições): {len(cpfs_dim):,}')
print(f'CPFs em comum: {len(cpfs_disparo & cpfs_dim):,}')
print(f'CPFs apenas em disparos: {len(cpfs_disparo - cpfs_dim):,}')
print(f'CPFs apenas na dimensão: {len(cpfs_dim - cpfs_disparo):,}')


Colunas da dimensão: ['telefone_ddi', 'telefone_ddd', 'telefone_numero', 'telefone_tipo', 'telefone_nacionalidade', 'telefone_qualidade', 'telefone_aparicoes', 'telefone_aparicoes_quantidade', 'telefone_proprietarios_quantidade', 'telefone_sistemas_quantidade', 'validacao_telefone']

CPFs únicos na base de disparos: 268,437
CPFs únicos na dimensão (via aparições): 1,221,517
CPFs em comum: 1,331
CPFs apenas em disparos: 267,106
CPFs apenas na dimensão: 1,220,186


---
## 11. Sumário de Achados

Após executar as células acima, preencha este sumário com os achados reais. Isso servirá como referência nos próximos notebooks.

| Verificação | Resultado Esperado / Ação Necessária |
|---|---|
| Shapes | `disparo`: ~393k, `telefone`: ~283k |
| Missing em chaves de join | Deve ser 0% em ambas as colunas |
| Status de disparo | Confirmado: `processing`, `sent`, `delivered`, `read`, `failed` |
| Match de chave | ~252k em comum; ~42k em disparos sem dim; ~30k na dim sem disparos |
| `telefone_aparicoes` | `list` de `dict`. Confirmar chaves: `id_sistema`, `cpf`, `proprietario_tipo`, `registro_data_atualizacao` |
| `telefone_tipo` | Verificar se há 'Fixo' — pode exigir filtro posterior |
| `telefone_qualidade` | ALTA / MEDIA / BAIXA — potencial feature para o algoritmo |
| Range de datas | Confirmar se há shift temporal (datas não devem ser reais) |

---

> **Próximo passo:** Após validar todos os pontos acima, prossiga para o notebook `01_eda_e_qualidade_fontes.ipynb`.